In [ ]:
%pip install -q -U langchain langgraph langchain-groq
%pip install -q -U langgraph-checkpoint-sqlite
%pip install -q -U google-search-results langchain-community

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY").strip()

In [ ]:
from typing import Annotated, TypedDict

from langchain_groq import ChatGroq

#StateGraph: Creates the workflow.
#START: Beginning of the workflow.
#END: Ending of the workflow.
from langgraph.graph import StateGraph, START, END

from langgraph.graph.message import add_messages #Adds new messages to the existing message history.


In [ ]:

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
from langchain_community.utilities import SerpAPIWrapper

/tmp/ipykernel_67363/1701457994.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SerpAPIWrapper


In [ ]:
@tool #tool decorator-llm can communicate
def add(a: float, b: float) -> float: #creating func with type hints
    """Add two numbers.""" #doc string
    return a + b

In [ ]:
print(add.name) #syntax:functionname.name
print(add.description)
print(add.args)

add
Add two numbers.
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


In [ ]:
print(add.args_schema.model_json_schema()) #schema-when we connect tool and llm we share schema

{'description': 'Add two numbers.', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


In [ ]:
@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

In [ ]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")

    return a / b

In [ ]:
search = SerpAPIWrapper()

@tool
def google_search(query: str) -> str:
    """Search Google for current information."""
    return search.run(query)

In [ ]:
tools = [add, subtract, multiply, divide,google_search]

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=500
)

In [ ]:
llm_with_tools = llm.bind_tools(tools)#when we connect tool and llm we share schema (llm dont call tool based on schema its decide)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
#LLm node
def call_model(state: State):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }


In [ ]:
#router
def router(state: State):
    last_message = state["messages"][-1]

    print("\n========== ROUTER ==========")
    print("Latest message type:", type(last_message).__name__)

    if last_message.tool_calls:
        print("Route selected: ai_response → tools")
        return "tools"

    print("Route selected: ai_response → end")
    return "end"

In [ ]:
#Workflow
workflow = StateGraph(State)

#Add node
workflow.add_node("llm", call_model)
workflow.add_node("tools", ToolNode(tools))

#Add edge
workflow.add_edge(START, "llm")

workflow.add_conditional_edges(
    "llm",
    router,
    {
        "tools": "tools",
        "end": END
    }
)

workflow.add_edge("tools", "llm")


In [ ]:

def print_messages(state):
    print("\n========== CONVERSATION ==========")

    for message in state["messages"]:
        if isinstance(message, HumanMessage):
            print("\nUSER RESPONSE:")
            print(message.content)

        elif isinstance(message, AIMessage):
            print("\nAI RESPONSE:")
            print(message.content)

            if message.tool_calls:
                print("AI TOOL CALL:")

                for tool_call in message.tool_calls:
                    print("Tool name:", tool_call["name"])
                    print("Tool arguments:", tool_call["args"])

        elif isinstance(message, ToolMessage):
            print("\nTOOL RESPONSE:")
            print(message.content)

    print("\n==================================")

In [ ]:
#user_input = "What is 25 multiplied by 16?"

#response = graph.invoke({
    #"messages": [
        #HumanMessage(content=user_input)
    #]
#})

#print_messages(response)

In [ ]:
test_response = llm_with_tools.invoke([
    HumanMessage(
        content="Search Google and give me today's top 5 news."
    )
])

print(test_response)
print("Tool calls:", test_response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': 'bsb7tfn8y', 'function': {'arguments': '{"query":"top 5 news today"}', 'name': 'google_search'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 545, 'total_tokens': 575, 'completion_time': 0.090171779, 'completion_tokens_details': None, 'prompt_time': 0.044940411, 'prompt_tokens_details': None, 'queue_time': 0.005619246, 'total_time': 0.13511219}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_c7e30c203c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0809c-e09a-7013-b95e-1adf09797327-0' tool_calls=[{'name': 'google_search', 'args': {'query': 'top 5 news today'}, 'id': 'bsb7tfn8y', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 545, 'output_tokens': 30, 'total_tokens': 575}
Tool calls: [{'name': 'google_search', 'args': {'query': 'top 5 news today'}, 'id': 'bsb7tfn8y', '

In [ ]:
with SqliteSaver.from_conn_string(
    "calculator_memory.sqlite"
) as memory:

    graph = workflow.compile(
        checkpointer=memory
    )

    thread_id = "calculator-user-1"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print("Graph compiled successfully")

    print("Calculator Chatbot Started")
    print("Type 'exit', 'quit', or 'stop' to end conversation")

    while True:
        user_input = input("User: ")

        if user_input.lower().strip() in [
            "exit",
            "quit",
            "stop"
        ]:
            print("Chatbot Stopped")
            break

        if not user_input.strip():
            print("Please enter a question.\n")
            continue

        response = graph.invoke(
            {
                "messages": [
                    HumanMessage(content=user_input)
                ]
            },
            config=config
        )

        print_messages(response)

Graph compiled successfully
Calculator Chatbot Started
Type 'exit', 'quit', or 'stop' to end conversation

========== ROUTER ==========
Latest message type: AIMessage
Route selected: ai_response → end

========== CONVERSATION ==========

USER RESPONSE:
4+2

AI RESPONSE:

AI TOOL CALL:
Tool name: add
Tool arguments: {'a': 4, 'b': 2}

TOOL RESPONSE:
6.0

AI RESPONSE:
4 + 2 = **6**

USER RESPONSE:
give me todays top 5 news

AI RESPONSE:
I don't have access to real-time information, so I cannot provide today's top 5 news headlines. My knowledge cutoff prevents me from browsing the internet or accessing current events.

For the latest news, I recommend checking reputable news sources such as:
*   Reuters
*   Associated Press (AP)
*   BBC News
*   CNN
*   Al Jazeera

Is there a specific topic or past event you'd like me to explain?

USER RESPONSE:
give me todays top 5 news

AI RESPONSE:
I do not have access to real-time information, so I cannot provide today's top 5 news headlines. My knowle